# DNN训练 - 使用PyTorch和MPS加速

本notebook使用PyTorch实现深度神经网络(DNN)，利用macOS的MPS加速进行训练。

## 模型架构
- 8层隐藏层：从512递减到32
- 激活函数：ReLU
- 优化器：Adam
- 目标变量：Ash_Deformation, Ash_Softening, Ash_Fluid


## 1. 导入库和设置


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import json
import time
from datetime import datetime
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']  # 支持中文
matplotlib.rcParams['axes.unicode_minus'] = False  # 正常显示负号

# JSON格式化函数（用于生成紧凑格式的JSON，特征重要性对象放在一行）
def format_json_compact(data, indent=2):
    """
    格式化JSON，对feature_importances使用紧凑格式（每个对象在一行）
    """
    def format_value(value, indent_level, indent_str):
        """递归格式化值"""
        if isinstance(value, dict):
            output_items = []
            current_indent = indent_str * indent_level
            
            for key, val in value.items():
                key_str = json.dumps(key, ensure_ascii=False)
                
                # 对 feature_importances 和 interaction_features 使用紧凑格式（每个对象在一行）
                if key == "feature_importances" or key == "interaction_features":
                    if val is None:
                        output_items.append(f'{current_indent}{key_str}: null')
                    elif isinstance(val, list):
                        items = []
                        next_indent = indent_str * (indent_level + 1)
                        for item in val:
                            if isinstance(item, dict) and 'importance' in item and 'feature_name' in item:
                                # 将 feature_importance 或 interaction_feature 对象格式化为同一行
                                compact_dict = json.dumps(item, ensure_ascii=False, separators=(', ', ': '))
                                items.append(next_indent + compact_dict)
                            else:
                                item_str = format_value(item, indent_level + 1, indent_str)
                                items.append(next_indent + item_str)
                        if not items:
                            output_items.append(f'{current_indent}{key_str}: []')
                        else:
                            feature_imp_str = '[\n' + ',\n'.join(items) + '\n' + current_indent + ']'
                            output_items.append(f'{current_indent}{key_str}: {feature_imp_str}')
                    else:
                        val_str = format_value(val, indent_level + 1, indent_str)
                        output_items.append(f'{current_indent}{key_str}: {val_str}')
                else:
                    # 递归处理其他字段
                    val_str = format_value(val, indent_level + 1, indent_str)
                    output_items.append(f'{current_indent}{key_str}: {val_str}')
            
            current_indent = indent_str * indent_level
            if not output_items:
                return '{}'
            return '{\n' + ',\n'.join(output_items) + '\n' + current_indent + '}'
        
        elif isinstance(value, list):
            if not value:
                return '[]'
            
            items = []
            current_indent = indent_str * indent_level
            next_indent = indent_str * (indent_level + 1)
            
            for item in value:
                item_str = format_value(item, indent_level + 1, indent_str)
                items.append(next_indent + item_str)
            
            return '[\n' + ',\n'.join(items) + '\n' + current_indent + ']'
        
        else:
            # 基本类型直接序列化
            return json.dumps(value, ensure_ascii=False)
    
    indent_str = ' ' * indent
    return format_value(data, 0, indent_str)



# 设置随机种子（确保结果可复现）
def set_seed(seed):
    """设置所有随机种子以确保结果可复现"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    # 设置PyTorch的确定性行为
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # 设置Python的随机种子
    import random
    random.seed(seed)
    # 设置环境变量（如果使用CUDA）
    import os
    os.environ['PYTHONHASHSEED'] = str(seed)

# 设备选择优先级：GPU (CUDA) > MPS > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"使用设备: {device} (GPU)")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"使用设备: {device} (MPS)")
else:
    device = torch.device("cpu")
    print(f"使用设备: {device} (CPU)")
    print("警告: GPU和MPS都不可用，将使用CPU训练（速度较慢）")


## 2. 配置参数


In [ ]:
# 数据配置路径
DATA_CONFIG_PATH = "/Users/m/Desktop/Gps/Dataset_split/dataset_onfiguration/43_20251203_235329/data_config_43_20251203_235329.json"

# 模型参数
RANDOM_SEEDS = [1]  # 使用10个不同的随机种子
USE_CROSS_VALIDATION = False  # 是否开启交叉验证
CV_FOLDS = 5  # 交叉验证折数（仅在USE_CROSS_VALIDATION=True时生效）
BATCH_SIZE = 128  # 增大批量大小以提升训练稳定性
LEARNING_RATE = 0.001  # 稍微提高初始学习率
MAX_EPOCHS = 2000
EARLY_STOPPING_PATIENCE = 30
VALIDATION_FRACTION = 0.15

# 正则化参数
DROPOUT_RATE = 0.2  # 添加dropout防止过拟合
WEIGHT_DECAY = 1e-5  # L2正则化（weight decay）

# DNN架构：8层，从512递减到32
HIDDEN_LAYERS = [512, 443, 374, 306, 237, 169, 100, 32]

# 是否使用Batch Normalization
USE_BATCH_NORM = True

print(f"DNN架构: {HIDDEN_LAYERS}")
print(f"隐藏层数量: {len(HIDDEN_LAYERS)}")
print(f"Dropout率: {DROPOUT_RATE}")
print(f"Weight Decay: {WEIGHT_DECAY}")
print(f"使用Batch Normalization: {USE_BATCH_NORM}")
print(f"随机种子列表: {RANDOM_SEEDS}")
print(f"运行次数: {len(RANDOM_SEEDS)}")
print(f"是否开启交叉验证: {USE_CROSS_VALIDATION}")
if USE_CROSS_VALIDATION:
    print(f"交叉验证折数: {CV_FOLDS}")


## 3. 定义DNN模型


In [ ]:
class DNN(nn.Module):
    def __init__(self, input_size, hidden_layers, dropout=0.0, use_batch_norm=False):
        super(DNN, self).__init__()
        
        layers = []
        prev_size = input_size
        
        # 构建隐藏层
        for i, hidden_size in enumerate(hidden_layers):
            layers.append(nn.Linear(prev_size, hidden_size))
            
            # 添加Batch Normalization（除了最后一层）
            if use_batch_norm and i < len(hidden_layers) - 1:
                layers.append(nn.BatchNorm1d(hidden_size))
            
            layers.append(nn.ReLU())
            
            # 添加Dropout（除了最后一层）
            if dropout > 0 and i < len(hidden_layers) - 1:
                layers.append(nn.Dropout(dropout))
            
            prev_size = hidden_size
        
        # 输出层
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
        
        # 权重初始化（Xavier初始化）
        self._initialize_weights()
    
    def _initialize_weights(self):
        """权重初始化"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.network(x).squeeze()

print("DNN模型定义完成（已添加Batch Normalization和权重初始化）")


## 4. 加载数据配置


In [ ]:
# 加载数据配置
with open(DATA_CONFIG_PATH, 'r', encoding='utf-8') as f:
    data_config = json.load(f)

train_set_path = data_config['train_set_path']
test_set_path = data_config['test_set_path']
selected_feature = data_config['selected_feature']
target_columns = data_config['target_column']
all_dataset_path = data_config.get('all_dataset_path', '')

print(f"训练集路径: {train_set_path}")
print(f"测试集路径: {test_set_path}")
print(f"特征数量: {len(selected_feature)}")
print(f"目标变量: {target_columns}")


## 5. 数据加载和预处理函数


In [ ]:
def load_data(target_name):
    """加载训练集和测试集"""
    train_df = pd.read_csv(train_set_path)
    test_df = pd.read_csv(test_set_path)
    
    # 提取特征（排除目标变量）
    feature_cols = [col for col in selected_feature if col != target_name]
    
    X_train = train_df[feature_cols].values.astype(np.float32)
    X_test = test_df[feature_cols].values.astype(np.float32)
    y_train = train_df[target_name].values.astype(np.float32)
    y_test = test_df[target_name].values.astype(np.float32)
    
    # 标准化特征
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler_X, feature_cols

print("数据加载函数定义完成")


## 6. 训练和评估函数


## 6. 训练和评估函数


In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs, batch_size, lr, patience, weight_decay=0.0, return_history=False):
    """训练模型
    
    参数:
        return_history: 如果为True，返回训练历史记录（用于绘制学习曲线）
    
    返回:
        model: 训练好的模型
        history: 如果return_history=True，返回包含train_losses和val_losses的字典
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    # 添加weight decay（L2正则化）
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    # 更积极的学习率衰减策略
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-6)
    
    # 转换为tensor
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.FloatTensor(y_val).to(device)
    
    # 创建数据加载器（设置generator以确保shuffle的可复现性）
    train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    # 创建generator以确保shuffle的可复现性
    generator = torch.Generator()
    generator.manual_seed(42)  # 固定generator的种子
    train_loader = torch.utils.data.DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
        generator=generator  # 使用固定的generator
    )
    
    # 记录训练历史
    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        num_batches = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            num_batches += 1
        
        # 计算平均训练损失
        avg_train_loss = train_loss / num_batches
        
        # 验证阶段
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor).item()
        
        # 记录损失
        if return_history:
            train_losses.append(avg_train_loss)
            val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        # 早停检查
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    if return_history:
        return model, {'train_losses': train_losses, 'val_losses': val_losses}
    else:
        return model

def calculate_metrics(y_true, y_pred):
    """计算评估指标"""
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mse, rmse, mae

def plot_learning_curves(history, save_path=None, title="学习曲线"):
    """绘制学习曲线
    
    参数:
        history: 包含'train_losses'和'val_losses'的字典
        save_path: 保存图片的路径（如果为None则不保存）
        title: 图表标题
    """
    train_losses = history['train_losses']
    val_losses = history['val_losses']
    epochs = range(1, len(train_losses) + 1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, 'b-', label='训练损失', linewidth=2)
    plt.plot(epochs, val_losses, 'r-', label='验证损失', linewidth=2)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss (MSE)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"学习曲线已保存至: {save_path}")
    
    plt.show()

def calculate_permutation_importance(model, X_test, y_test, feature_names, n_repeats=5, random_seed=42):
    """
    计算排列特征重要性（Permutation Feature Importance）
    
    通过随机打乱某一列特征，观察MSE的变化来评估特征重要性
    
    参数:
        model: 训练好的PyTorch模型
        X_test: 测试集特征（numpy数组）
        y_test: 测试集标签（numpy数组）
        feature_names: 特征名称列表
        n_repeats: 每个特征重复打乱的次数，默认5
        random_seed: 随机种子
    
    返回:
        feature_importance_list: 特征重要性列表，格式为 [{"importance": float, "feature_name": str}, ...]
    """
    model.eval()
    
    # 设置随机种子
    np.random.seed(random_seed)
    
    # 转换为tensor并计算原始性能
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    with torch.no_grad():
        y_pred_original = model(X_test_tensor).cpu().numpy()
    
    original_mse = mean_squared_error(y_test, y_pred_original)
    
    # 初始化特征重要性数组
    feature_importances = np.zeros(X_test.shape[1])
    
    # 对每个特征计算重要性
    for i in range(X_test.shape[1]):
        importance_scores = []
        
        for _ in range(n_repeats):
            # 复制测试集
            X_test_permuted = X_test.copy()
            
            # 打乱第i个特征的值
            permuted_indices = np.random.permutation(len(X_test_permuted))
            X_test_permuted[:, i] = X_test_permuted[permuted_indices, i]
            
            # 使用打乱后的数据进行预测
            X_test_permuted_tensor = torch.FloatTensor(X_test_permuted).to(device)
            with torch.no_grad():
                y_pred_permuted = model(X_test_permuted_tensor).cpu().numpy()
            
            # 计算打乱后的MSE
            permuted_mse = mean_squared_error(y_test, y_pred_permuted)
            
            # 重要性 = 打乱后的MSE - 原始MSE（MSE增加越多，特征越重要）
            importance_scores.append(permuted_mse - original_mse)
        
        # 取平均值
        feature_importances[i] = np.mean(importance_scores)
    
    # 创建特征重要性列表
    feature_importance_list = [
        {"importance": float(imp), "feature_name": name}
        for imp, name in zip(feature_importances, feature_names)
    ]
    
    # 按重要性降序排序
    feature_importance_list.sort(key=lambda x: x["importance"], reverse=True)
    
    return feature_importance_list

print("训练和评估函数定义完成（已添加学习曲线功能）")
print("特征重要性计算函数定义完成（Permutation Importance）")


## 7. 交叉验证函数


In [ ]:
def cross_validate(X_train, y_train, input_size, hidden_layers, cv_folds, random_seed):
    """K折交叉验证"""
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=random_seed)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"  交叉验证 Fold {fold + 1}/{cv_folds}...")
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # 创建模型（使用新的正则化参数）
        set_seed(random_seed + fold)
        model = DNN(input_size, hidden_layers, dropout=DROPOUT_RATE, use_batch_norm=USE_BATCH_NORM)
        
        # 训练模型（添加weight_decay）
        model = train_model(
            model, X_tr, y_tr, X_val, y_val,
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LEARNING_RATE,
            patience=EARLY_STOPPING_PATIENCE,
            weight_decay=WEIGHT_DECAY
        )
        
        # 评估
        model.eval()
        with torch.no_grad():
            X_val_tensor = torch.FloatTensor(X_val).to(device)
            y_val_pred = model(X_val_tensor).cpu().numpy()
        
        r2, mse, rmse, mae = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append({'r2': r2, 'mse': mse, 'rmse': rmse, 'mae': mae})
    
    # 计算均值和标准差
    cv_summary = {
        'r2_mean': float(np.mean([m['r2'] for m in fold_metrics])),
        'r2_std': float(np.std([m['r2'] for m in fold_metrics])),
        'mse_mean': float(np.mean([m['mse'] for m in fold_metrics])),
        'mse_std': float(np.std([m['mse'] for m in fold_metrics])),
        'rmse_mean': float(np.mean([m['rmse'] for m in fold_metrics])),
        'rmse_std': float(np.std([m['rmse'] for m in fold_metrics])),
        'mae_mean': float(np.mean([m['mae'] for m in fold_metrics])),
        'mae_std': float(np.std([m['mae'] for m in fold_metrics]))
    }
    
    return cv_summary

print("交叉验证函数定义完成")


## 8. 主训练流程


In [ ]:
# 生成时间戳
base_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"dnn_result_{base_timestamp}"
os.makedirs(output_dir, exist_ok=True)

print(f"结果将保存至: {output_dir}")
print("=" * 80)


In [ ]:
# 遍历每个目标变量
all_results = {}

for target_name in target_columns:
    print(f"\n处理目标变量: {target_name}")
    print("=" * 80)
    
    # 加载数据
    X_train, X_test, y_train, y_test, scaler_X, feature_cols = load_data(target_name)
    input_size = X_train.shape[1]
    
    print(f"特征数量: {input_size}")
    print(f"训练集样本数: {X_train.shape[0]}")
    print(f"测试集样本数: {X_test.shape[0]}")
    
    # 创建目标变量目录
    target_dir = os.path.join(output_dir, target_name)
    os.makedirs(target_dir, exist_ok=True)
    
    # 遍历每个随机种子
    for seed_idx, random_seed in enumerate(RANDOM_SEEDS, 1):
        print(f"\n{'='*80}")
        print(f"运行 {seed_idx}/{len(RANDOM_SEEDS)} - 随机种子: {random_seed}")
        print(f"{'='*80}")
        
        # 记录开始时间
        start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # 设置随机种子
        set_seed(random_seed)
        
        # 交叉验证（根据开关决定是否执行）
        if USE_CROSS_VALIDATION:
            print(f"\n开始 {CV_FOLDS} 折交叉验证...")
            cv_summary = cross_validate(X_train, y_train, input_size, HIDDEN_LAYERS, CV_FOLDS, random_seed)
        else:
            print(f"\n跳过交叉验证（USE_CROSS_VALIDATION=False）")
            # 创建一个占位符的cv_summary
            cv_summary = {
                'r2_mean': float('nan'),
                'r2_std': float('nan'),
                'mse_mean': float('nan'),
                'mse_std': float('nan'),
                'rmse_mean': float('nan'),
                'rmse_std': float('nan'),
                'mae_mean': float('nan'),
                'mae_std': float('nan')
            }
        
        # 最终模型训练（使用全部训练集）
        print(f"\n训练最终模型（使用全部训练集）...")
        
        # 在划分验证集之前设置随机种子，确保划分可复现
        set_seed(random_seed)
        
        # 划分验证集用于早停
        val_size = int(len(X_train) * VALIDATION_FRACTION)
        indices = np.random.permutation(len(X_train))
        train_indices = indices[val_size:]
        val_indices = indices[:val_size]
        
        X_tr_final = X_train[train_indices]
        X_val_final = X_train[val_indices]
        y_tr_final = y_train[train_indices]
        y_val_final = y_train[val_indices]
        
        # 再次设置随机种子，确保模型初始化可复现
        set_seed(random_seed)
        final_model = DNN(input_size, HIDDEN_LAYERS, dropout=DROPOUT_RATE, use_batch_norm=USE_BATCH_NORM)
        
        train_start = time.time()
        # 训练模型并记录历史（用于绘制学习曲线，仅对第一个随机种子保存）
        if seed_idx == 1:
            final_model, training_history = train_model(
                final_model, X_tr_final, y_tr_final, X_val_final, y_val_final,
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                lr=LEARNING_RATE,
                patience=EARLY_STOPPING_PATIENCE,
                weight_decay=WEIGHT_DECAY,
                return_history=True  # 仅第一个随机种子保存学习曲线
            )
            # 绘制并保存学习曲线（仅第一个随机种子）
            learning_curve_path = os.path.join(target_dir, f"learning_curve_{target_name}_{base_timestamp}.png")
            plot_learning_curves(
                training_history, 
                save_path=learning_curve_path,
                title=f"{target_name} - 学习曲线 (随机种子={random_seed})"
            )
        else:
            final_model = train_model(
                final_model, X_tr_final, y_tr_final, X_val_final, y_val_final,
                epochs=MAX_EPOCHS,
                batch_size=BATCH_SIZE,
                lr=LEARNING_RATE,
                patience=EARLY_STOPPING_PATIENCE,
                weight_decay=WEIGHT_DECAY,
                return_history=False
            )
        training_time = time.time() - train_start
        
        # 评估训练集和测试集
        final_model.eval()
        with torch.no_grad():
            X_train_tensor = torch.FloatTensor(X_train).to(device)
            X_test_tensor = torch.FloatTensor(X_test).to(device)
            y_train_pred = final_model(X_train_tensor).cpu().numpy()
            y_test_pred = final_model(X_test_tensor).cpu().numpy()
        
        train_r2, train_mse, train_rmse, train_mae = calculate_metrics(y_train, y_train_pred)
        test_r2, test_mse, test_rmse, test_mae = calculate_metrics(y_test, y_test_pred)
        
        # DNN模型不支持传统的feature_importances_，设置为None
        feature_importances = None
        
        # 记录结束时间
        end_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # 计算训练时长（秒、分钟、小时）
        training_duration = {
            'seconds': float(training_time),
            'minutes': float(training_time / 60.0),
            'hours': float(training_time / 3600.0)
        }
        
        # 构建结果
        result = {
            'model_name': 'DNN',
            'cv_metrics': {
                'folds': CV_FOLDS if USE_CROSS_VALIDATION else 0,
                'cross_validation': cv_summary
            },
            'train_metrics': {
                'r2': float(train_r2),
                'mse': float(train_mse),
                'rmse': float(train_rmse),
                'mae': float(train_mae)
            },
            'test_metrics': {
                'r2': float(test_r2),
                'mse': float(test_mse),
                'rmse': float(test_rmse),
                'mae': float(test_mae)
            },
            'training_duration': training_duration,
            'feature_importances': feature_importances if feature_importances is not None else None
        }
        
        # 构建完整的JSON结构
        result_data = {
            "experiment_info": {
                "start_time": start_time,
                "end_time": end_time,
                "random_seeds": random_seed,
                "all_dataset_path": all_dataset_path if all_dataset_path else train_set_path,
                "train_file_path": train_set_path,
                "test_file_path": test_set_path,
                "train_set_size": X_train.shape[0],
                "test_set_size": X_test.shape[0],
                "feature_names": selected_feature,
                "target_variable": target_name
            },
            "baseline_info": {
                "baseline_models": [result]
            }
        }
        
        # 保存结果（文件名格式：baseline_{random_seed}_{timestamp}.json）
        filename = f"baseline_{random_seed}_{base_timestamp}.json"
        filepath = os.path.join(target_dir, filename)
        
        # 使用紧凑格式保存JSON（特征重要性对象放在一行）
        json_str = format_json_compact(result_data, indent=2)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(json_str)
        
        # 存储结果（仅保存第一个随机种子的结果用于汇总）
        if seed_idx == 1:
            all_results[target_name] = result_data
        
        # 打印结果摘要
        print(f"\n结果摘要 (随机种子={random_seed}):")
        if USE_CROSS_VALIDATION:
            print(f"  CV_R²: {cv_summary['r2_mean']:.6f}±{cv_summary['r2_std']:.6f}")
        print(f"  Train_R²: {train_r2:.6f}")
        print(f"  Test_R²: {test_r2:.6f}")
        print(f"  训练时间: {training_time:.2f}秒")
        print(f"结果已保存至: {filepath}")
        print("-" * 80)
    
    print(f"\n✓ {target_name} 的所有 {len(RANDOM_SEEDS)} 次运行完成！")
    print("=" * 80)

print("\n所有目标变量处理完成！")


## 9. 结果汇总


In [ ]:
# 打印所有结果汇总（统计所有随机种子的结果）
print("\n" + "=" * 80)
print("结果汇总（所有随机种子的统计）")
print("=" * 80)

# 统计所有随机种子的结果
for target_name in target_columns:
    print(f"\n{target_name}:")
    print("-" * 80)
    
    # 读取该目标变量的所有结果文件
    target_dir = os.path.join(output_dir, target_name)
    all_cv_r2 = []
    all_test_r2 = []
    all_test_rmse = []
    all_test_mae = []
    all_training_times = []
    
    for random_seed in RANDOM_SEEDS:
        filename = f"baseline_{random_seed}_{base_timestamp}.json"
        filepath = os.path.join(target_dir, filename)
        
        if os.path.exists(filepath):
            with open(filepath, 'r', encoding='utf-8') as f:
                result_data = json.load(f)
            
            baseline = result_data['baseline_info']['baseline_models'][0]
            cv = baseline['cv_metrics']['cross_validation']
            test = baseline['test_metrics']
            
            # 只有在开启交叉验证时才收集CV指标
            if USE_CROSS_VALIDATION and not np.isnan(cv['r2_mean']):
                all_cv_r2.append(cv['r2_mean'])
            all_test_r2.append(test['r2'])
            all_test_rmse.append(test['rmse'])
            all_test_mae.append(test['mae'])
            # 兼容旧格式（training_time）和新格式（training_duration）
            if 'training_duration' in baseline:
                all_training_times.append(baseline['training_duration']['seconds'])
            elif 'training_time' in baseline:
                all_training_times.append(baseline['training_time'])
    
    # 计算统计信息
    if USE_CROSS_VALIDATION and all_cv_r2:
        print(f"  CV_R² (平均): {np.mean(all_cv_r2):.6f}±{np.std(all_cv_r2):.6f}")
        print(f"  CV_R² (范围): [{np.min(all_cv_r2):.6f}, {np.max(all_cv_r2):.6f}]")
    
    if all_test_r2:
        print(f"  Test_R² (平均): {np.mean(all_test_r2):.6f}±{np.std(all_test_r2):.6f}")
        print(f"  Test_R² (范围): [{np.min(all_test_r2):.6f}, {np.max(all_test_r2):.6f}]")
        print(f"  Test_RMSE (平均): {np.mean(all_test_rmse):.6f}±{np.std(all_test_rmse):.6f}")
        print(f"  Test_MAE (平均): {np.mean(all_test_mae):.6f}±{np.std(all_test_mae):.6f}")
        print(f"  训练时间 (平均): {np.mean(all_training_times):.2f}秒")
        print(f"  总训练时间: {np.sum(all_training_times):.2f}秒")
    
    # 统计实际生成的文件数
    json_files = [f for f in os.listdir(target_dir) if f.endswith('.json')]
    print(f"  生成文件数: {len(json_files)} 个")

print(f"\n所有结果保存在: {output_dir}")
print(f"每个目标变量下共有 {len(RANDOM_SEEDS)} 个结果文件（对应随机种子 {RANDOM_SEEDS}）")
